A Python notebook to compare results based on same enforced spans across three different datasets with our NER types.


BioDivNER: 
```shell
"DATA/LABEL_STUDIO/LS_BIODIVNER/project-34-at-2025-12-08-13-44-2171ef2d.json"
```

ClimateIE: 
```shell
"DATA/LABEL_STUDIO/LS_CLIMATEIE/project-35-at-2025-12-05-07-53-b0139f8e.json"
```

Climate Change NER (IBM): 
```shell
"DATA/LABEL_STUDIO/LS_IBMCCNER/project-36-at-2025-12-08-13-43-e4ba28f1.json"
```

In [31]:
from dataset_processing import cwed4eta_process_json_file
import json 

import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [42]:

def generate_label_distribution_report(
    user_dataset, 
    original_dataset, 
    output_dir="RESULTS/OTHER_DATASETS_ANNOTATIONS/OVERLAPS/",
    og_dataset_name="BioDivNER",
    map_dataset_name="CliReNER",
    filename_prefix="biodivner_distribution"
):
    """
    Compares label distributions between two annotated datasets with fixed entity spans.
    Generates CSV tables and a Heatmap plot.
    
    Args:
        user_dataset (list): List of dicts (User version, flat structure).
        original_dataset (list): List of dicts (Original version, nested 'data' key).
        output_dir (str): Path to save results.
        filename_prefix (str): Prefix for generated files.
    """
    
    # 1. Setup Directory
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created directory: {output_dir}")

    # 2. Extract and Align Labels
    mapping_data = []
    
    # We assume lists are sorted and aligned by document ID
    for user_doc, og_doc_wrapper in zip(user_dataset, original_dataset):
        
        # Handle structural differences based on your provided schema
        # Original: wrapped in ["data"]
        # User: direct access
        og_entities = og_doc_wrapper.get("data", {}).get("entities", [])
        user_entities = user_doc.get("entities", [])

        # Create Lookup for Original: Key = (start, end), Value = Label
        og_lookup = {
            (e['start'], e['end']): e['label'] 
            for e in og_entities
        }
        
        # Match User entities to Original
        for user_ent in user_entities:
            span_key = (user_ent['start'], user_ent['end'])
            
            if span_key in og_lookup:
                mapping_data.append({
                    "Original_Label": og_lookup[span_key],
                    "User_Label": user_ent['label']
                })

    if not mapping_data:
        print("No matching spans found between datasets.")
        return

    df_map = pd.DataFrame(mapping_data)

    # 3. Generate Matrices
    
    # A. Raw Counts
    count_matrix = pd.crosstab(
        df_map["Original_Label"], 
        df_map["User_Label"]
    )
    
    # B. Percentages (Row-wise normalization)
    # This shows: Of all [Original_Row] entities, X% became [User_Col]
    pct_matrix = pd.crosstab(
        df_map["Original_Label"], 
        df_map["User_Label"], 
        normalize='index'
    )
    pct_matrix = (pct_matrix * 100).round(2)

    # 4. Save Tables to CSV
    count_csv_path = os.path.join(output_dir, f"{filename_prefix}_counts.csv")
    pct_csv_path = os.path.join(output_dir, f"{filename_prefix}_percentages.csv")
    
    count_matrix.to_csv(count_csv_path)
    pct_matrix.to_csv(pct_csv_path)
    
    print(f"Saved Count Table: {count_csv_path}")
    print(f"Saved Percentage Table: {pct_csv_path}")

    # 5. Generate and Save Heatmap Plot
    # Dynamic sizing: 
    # Width = number of user labels * 0.8 (min 10)
    # Height = number of original labels * 0.8 (min 8)
    h_w = max(10, len(pct_matrix.columns) * 0.8)
    h_h = max(8, len(pct_matrix.index) * 0.8)

    plt.figure(figsize=(h_w, h_h))
    
    sns.heatmap(
        pct_matrix, 
        annot=True,         # Write the percentage in the box
        fmt='.1f',          # Format to 1 decimal place
        cmap="Oranges",      # Yellow-Green-Blue colormap
        linewidths=.5,      # Space between squares
        cbar_kws={'label': 'Percentage (%)'}
    )
    
    # plt.title("Label Transition Matrix: Original vs User Annotations")
    plt.ylabel(f"{og_dataset_name} Lables")
    plt.xlabel(f"{map_dataset_name} Labels")
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()

    plot_path = os.path.join(output_dir, f"{filename_prefix}_heatmap.png")
    plt.savefig(plot_path, dpi=300)
    plt.close() # Close plot to free memory
    
    print(f"Saved Heatmap Plot: {plot_path}")


In [40]:
BioDivNER_DIR =  "DATA/LABEL_STUDIO/LS_BIODIVNER/project-34-at-2025-12-08-13-44-2171ef2d.json"
ClimateIE_DIR = "DATA/LABEL_STUDIO/LS_CLIMATEIE/project-35-at-2025-12-05-07-53-b0139f8e.json"
Climate_Change_NER_DIR = "DATA/LABEL_STUDIO/LS_IBMCCNER/project-36-at-2025-12-08-13-43-e4ba28f1.json"

### BioDivNER

In [43]:
biodivner = cwed4eta_process_json_file(BioDivNER_DIR)
with open("RESULTS/OTHER_DATASETS_ANNOTATIONS/golden_50_Env_Loc_Org_Mat_Phe_Qua.json", "r") as f:
    biodivner_og = json.load(f)

generate_label_distribution_report(biodivner, biodivner_og)

Saved Count Table: RESULTS/OTHER_DATASETS_ANNOTATIONS/OVERLAPS/biodivner_distribution_counts.csv
Saved Percentage Table: RESULTS/OTHER_DATASETS_ANNOTATIONS/OVERLAPS/biodivner_distribution_percentages.csv
Saved Heatmap Plot: RESULTS/OTHER_DATASETS_ANNOTATIONS/OVERLAPS/biodivner_distribution_heatmap.png


### ClimateIE

In [44]:
climateie = cwed4eta_process_json_file(ClimateIE_DIR)
with open("RESULTS/OTHER_DATASETS_ANNOTATIONS/golden_50_Mod_WeaEve_Ins_OceCir_Tel_Pro_NatHaz_Pro_Loc_Var.json", "r") as f:
    climateie_og = json.load(f)

generate_label_distribution_report(climateie, climateie_og, og_dataset_name="ClimateIE", filename_prefix="climatie_distribution")

Saved Count Table: RESULTS/OTHER_DATASETS_ANNOTATIONS/OVERLAPS/climatie_distribution_counts.csv
Saved Percentage Table: RESULTS/OTHER_DATASETS_ANNOTATIONS/OVERLAPS/climatie_distribution_percentages.csv
Saved Heatmap Plot: RESULTS/OTHER_DATASETS_ANNOTATIONS/OVERLAPS/climatie_distribution_heatmap.png


### Climate Change NER (IBM)


In [45]:
ibmccner = cwed4eta_process_json_file(Climate_Change_NER_DIR)
with open("RESULTS/OTHER_DATASETS_ANNOTATIONS/golden_50_Cli_Cli_Cli_Cli_Cli_Cli_Cli_Cli_Cli_Cli_Cli_Cli_Cli.json", "r") as f:
    ibmccner_og = json.load(f)

generate_label_distribution_report(ibmccner, ibmccner_og, og_dataset_name="Climate Change NER", filename_prefix="ibmccner_distribution")

Saved Count Table: RESULTS/OTHER_DATASETS_ANNOTATIONS/OVERLAPS/ibmccner_distribution_counts.csv
Saved Percentage Table: RESULTS/OTHER_DATASETS_ANNOTATIONS/OVERLAPS/ibmccner_distribution_percentages.csv
Saved Heatmap Plot: RESULTS/OTHER_DATASETS_ANNOTATIONS/OVERLAPS/ibmccner_distribution_heatmap.png
